# Cifras complementarias de la tesis

Este notebook calcula las cifras de la tesis que no calcula ningún otro archivo de esta carpeta. Las demás
salen de `analisis_corrida.py` (`g150p30m12touuniP2c2600s42.analisis.json`), `robustez.py`
(`robustez/g150p30m12touuniP2c2600s42.json`), `saturacion.py`, `aceleracion.py` y `video/datos.py`.

Todas las cifras corresponden a la ejecución `g150p30m12touuniP2c2600s42` y se calculan con pandas a partir de los
archivos que dejan la ejecución y los análisis:

| Archivo | Contenido |
|---|---|
| `test.net.xml` | red con la configuración de netconvert |
| `map.osm` | exportación de OpenStreetMap de la que sale la red |
| `g150p30m12touuniP2c2600s42.evaluations.csv` | bitácora de las 4380 evaluaciones del algoritmo |
| `detail/g150p30m12touuniP2c2600s42.{baseline,best}.tripinfo.xml` | registro de cada vehículo con netconvert y con la mejor solución |
| `detail/g150p30m12touuniP2c2600s42.{baseline,best}.edgedata.xml` | tiempo perdido por arista con netconvert y con la mejor solución |
| `outputs/an_g150p30m12touuniP2c2600s42_u{20..120}.xml` y `.stats.xml` | simulaciones del barrido de ciclo uniforme |
| `g150p30m12touuniP2c2600s42.analisis.json` | se lee solo para contrastar valores ya calculados |
| `robustez/g150p30m12touuniP2c2600s42.json` | las 105 simulaciones de robustez |

El notebook no escribe ningún archivo: los resultados quedan en las salidas de cada celda. Cada sección termina con
una tabla que compara la cifra que aparece en la tesis con el valor calculado, redondeado a los mismos decimales.

In [1]:
import json
import sys
import xml.etree.ElementTree as ET
from collections import defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

print("Python:", sys.executable)
print("pandas", pd.__version__, "| numpy", np.__version__)

RUN = "g150p30m12touuniP2c2600s42"
REGISTRO = []


def verificar(seccion, cifra, tesis, calculado, decimales):
    """compara una cifra de la tesis con el valor calculado, redondeado a los decimales con que aparece en la tesis"""
    redondeado = round(float(calculado), decimales)
    if decimales == 0:
        redondeado = int(redondeado)
    REGISTRO.append({"seccion": seccion, "cifra": cifra, "tesis": tesis, "calculado": float(calculado),
                     "redondeado": redondeado, "coincide": bool(redondeado == tesis)})


def tabla(seccion):
    return pd.DataFrame([r for r in REGISTRO if r["seccion"] == seccion]).drop(columns="seccion")

Python: C:\Users\noone\Documents\GitHub\upmega\.venv\Scripts\python.exe
pandas 2.3.1 | numpy 2.3.1


## 1. Metodología: semáforos y fases de la red

Sección "Representación del individuo". Se leen los programas (`tlLogic`) que netconvert asigna en `test.net.xml`.
Cada fase se clasifica con la misma regla que `Controller.phase_kind`: es verde si algún movimiento tiene `G` o `g`,
ámbar si no tiene verde y tiene `y`, y roja en otro caso. El ciclo mínimo de cada semáforo es la suma de sus ámbares
más 7 s por fase verde y 3 s por fase roja, como en `Controller.min_cycle`.

In [2]:
net = ET.parse("test.net.xml").getroot()


def tipo_fase(estado):
    if "G" in estado or "g" in estado:
        return "verde"
    if "y" in estado:
        return "ambar"
    return "rojo"


filas = []
for tl in net.findall("tlLogic"):
    fases = [(float(p.get("duration")), p.get("state")) for p in tl.findall("phase")]
    tipos = [tipo_fase(s) for _, s in fases]
    ambar_s = sum(int(round(d)) for (d, _), t in zip(fases, tipos) if t == "ambar")
    minimos = sum(7 if t == "verde" else 3 for t in tipos if t != "ambar")
    filas.append({"semaforo": tl.get("id"), "fases": len(fases), "ambares": tipos.count("ambar"),
                  "no_ambar": len(fases) - tipos.count("ambar"), "patron": "-".join(tipos),
                  "ciclo_s": sum(d for d, _ in fases), "desfase_s": float(tl.get("offset", 0)),
                  "ciclo_minimo_s": ambar_s + minimos})
sem = pd.DataFrame(filas).set_index("semaforo")
display(sem.head())
display(sem.groupby(["fases", "patron"]).size().rename("semaforos").to_frame())
display(sem.groupby("fases").ciclo_minimo_s.agg(["min", "max"]))

,fases,ambares,no_ambar,patron,ciclo_s,desfase_s,ciclo_minimo_s
semaforo,,,,,,,
1438391835,3,1,2,verde-ambar-rojo,90.0,0.0,16
1564620801,4,2,2,verde-ambar-verde-ambar,90.0,0.0,20
1564620830,3,1,2,verde-ambar-rojo,90.0,0.0,13
1903163086,3,1,2,verde-ambar-rojo,90.0,0.0,16
1903163087,3,1,2,verde-ambar-rojo,90.0,0.0,16


,,semaforos
fases,patron,
3,verde-ambar-rojo,54
4,verde-ambar-verde-ambar,56


,min,max
fases,,
3,13,16
4,20,26


In [3]:
S = "1. Red"
rangos = sem.groupby("fases").ciclo_minimo_s.agg(["min", "max"])
verificar(S, "intersecciones semaforizadas", 110, len(sem), 0)
verificar(S, "fases en total", 386, sem.fases.sum(), 0)
verificar(S, "fases de ambar retiradas del espacio de busqueda", 166, sem.ambares.sum(), 0)
verificar(S, "pesos de reparto (fases que no son ambar)", 220, sem.no_ambar.sum(), 0)
verificar(S, "intersecciones con plan de cuatro fases", 56, (sem.fases == 4).sum(), 0)
verificar(S, "intersecciones con plan de tres fases", 54, (sem.fases == 3).sum(), 0)
verificar(S, "intersecciones con ciclo de 90 s en netconvert", 110, (sem.ciclo_s == 90).sum(), 0)
verificar(S, "intersecciones con desfase cero en netconvert", 110, (sem.desfase_s == 0).sum(), 0)
verificar(S, "ciclo minimo de tres fases: desde (s)", 13, rangos.loc[3, "min"], 0)
verificar(S, "ciclo minimo de tres fases: hasta (s)", 16, rangos.loc[3, "max"], 0)
verificar(S, "ciclo minimo de cuatro fases: desde (s)", 20, rangos.loc[4, "min"], 0)
verificar(S, "ciclo minimo de cuatro fases: hasta (s)", 26, rangos.loc[4, "max"], 0)
tabla(S)

,cifra,tesis,calculado,redondeado,coincide
0,intersecciones semaforizadas,110,110.0,110,True
1,fases en total,386,386.0,386,True
2,fases de ambar retiradas del espacio de busqueda,166,166.0,166,True
3,pesos de reparto (fases que no son ambar),220,220.0,220,True
4,intersecciones con plan de cuatro fases,56,56.0,56,True
5,intersecciones con plan de tres fases,54,54.0,54,True
6,intersecciones con ciclo de 90 s en netconvert,110,110.0,110,True
7,intersecciones con desfase cero en netconvert,110,110.0,110,True
8,ciclo minimo de tres fases: desde (s),13,13.0,13,True
9,ciclo minimo de tres fases: hasta (s),16,16.0,16,True


### Entradas y ubicación de los semáforos de tres fases

Para cada semáforo se cuentan las aristas de entrada que controla (las conexiones con su `tl`) y se revisa el nudo
de la red en que se ubica:

- No hay cruce vehicular si al nudo llega y sale un solo corredor: como máximo dos aristas de entrada, dos de salida y dos nudos vecinos, contando solo aristas que admiten vehículos particulares.
- Hay calle peatonal transversal si el nudo pertenece en `map.osm` a una vía peatonal (`highway` igual a `pedestrian`, `footway`, `path` o `steps`).
- Una entrada es de un solo sentido si no existe la arista opuesta: netconvert nombra los dos sentidos de una calle como `<id>` y `-<id>`.

In [4]:
aristas_red = {}
for e in net.findall("edge"):
    if e.get("function") == "internal":
        continue
    carriles = e.findall("lane")
    vehicular = any("passenger" not in (c.get("disallow") or "") and
                    (c.get("allow") is None or "passenger" in c.get("allow")) for c in carriles)
    aristas_red[e.get("id")] = {"desde": e.get("from"), "hasta": e.get("to"), "vehicular": vehicular}

entradas = defaultdict(set)
for c in net.findall("connection"):
    if c.get("tl"):
        entradas[c.get("tl")].add(c.get("from"))

llegan, salen = defaultdict(list), defaultdict(list)
for eid, a in aristas_red.items():
    if a["vehicular"]:
        salen[a["desde"]].append(eid)
        llegan[a["hasta"]].append(eid)


def sentido_unico(eid):
    base = eid.lstrip("-")
    opuesta = base if eid.startswith("-") else "-" + base
    return opuesta not in aristas_red


def nodo(tl):
    nodos = {aristas_red[e]["hasta"] for e in entradas[tl]}
    assert len(nodos) == 1, tl
    return nodos.pop()


def sin_cruce_vehicular(j):
    vecinos = {aristas_red[e]["desde"] for e in llegan[j]} | {aristas_red[e]["hasta"] for e in salen[j]}
    return len(llegan[j]) <= 2 and len(salen[j]) <= 2 and len(vecinos) <= 2


osm = ET.parse("map.osm").getroot()
vias_del_nodo = defaultdict(set)
for w in osm.findall("way"):
    etiquetas = {t.get("k"): t.get("v") for t in w.findall("tag")}
    if "highway" in etiquetas:
        for nd in w.findall("nd"):
            vias_del_nodo[nd.get("ref")].add(etiquetas["highway"])
PEATONALES = {"pedestrian", "footway", "path", "steps"}

sem["entradas"] = [len(entradas[t]) for t in sem.index]
sem["entradas_sentido_unico"] = [sum(sentido_unico(e) for e in entradas[t]) for t in sem.index]
sem["nodo"] = [nodo(t) for t in sem.index]
sem["sin_cruce_vehicular"] = [sin_cruce_vehicular(j) for j in sem.nodo]
sem["calle_peatonal"] = [bool(PEATONALES & vias_del_nodo[j]) for j in sem.nodo]


def ubicacion(r):
    if r.fases == 4:
        return "cuatro fases"
    if not r.sin_cruce_vehicular:
        return "tres fases: en un cruce de calles"
    if r.calle_peatonal:
        return "tres fases: en el cruce con una calle peatonal"
    return "tres fases: a mitad de cuadra, sin calle transversal"


sem["ubicacion"] = sem.apply(ubicacion, axis=1)
display(pd.crosstab(sem.fases, sem.entradas, margins=True))
display(sem.ubicacion.value_counts().to_frame("semaforos"))
display(sem[sem.ubicacion != "cuatro fases"].groupby("ubicacion")[["entradas", "calle_peatonal"]].agg(["min", "max"]))

entradas,1,2,All
fases,,,
3,54,0,54
4,0,56,56
All,54,56,110


,semaforos
ubicacion,
cuatro fases,56
"tres fases: a mitad de cuadra, sin calle transversal",48
tres fases: en el cruce con una calle peatonal,5
tres fases: en un cruce de calles,1


entradas     calle_peatonal       
                                                          min max            min    max
ubicacion                                                                              
tres fases: a mitad de cuadra, sin calle transversal        1   1          False  False
tres fases: en el cruce con una calle peatonal              1   1           True   True
tres fases: en un cruce de calles                           1   1           True   True

In [5]:
cuatro = sem[sem.fases == 4]
print("Cuatro fases: semaforos cuyas dos entradas son de un solo sentido:",
      int((cuatro.entradas_sentido_unico == 2).sum()), "de", len(cuatro),
      "(la tesis dice 'en su mayoria')")
verificar(S, "tres fases que controlan una sola via de entrada", 54,
          ((sem.fases == 3) & (sem.entradas == 1)).sum(), 0)
verificar(S, "tres fases a mitad de cuadra, sin calle transversal", 48,
          (sem.ubicacion == "tres fases: a mitad de cuadra, sin calle transversal").sum(), 0)
verificar(S, "genes por hijo que cambia la mutacion (440 x 0.02)", 9, 440 * 0.02, 0)
tabla(S)

Cuatro fases: semaforos cuyas dos entradas son de un solo sentido: 56 de 56 (la tesis dice 'en su mayoria')


,cifra,tesis,calculado,redondeado,coincide
0,intersecciones semaforizadas,110,110.0,110,True
1,fases en total,386,386.0,386,True
2,fases de ambar retiradas del espacio de busqueda,166,166.0,166,True
3,pesos de reparto (fases que no son ambar),220,220.0,220,True
4,intersecciones con plan de cuatro fases,56,56.0,56,True
5,intersecciones con plan de tres fases,54,54.0,54,True
6,intersecciones con ciclo de 90 s en netconvert,110,110.0,110,True
7,intersecciones con desfase cero en netconvert,110,110.0,110,True
8,ciclo minimo de tres fases: desde (s),13,13.0,13,True
9,ciclo minimo de tres fases: hasta (s),16,16.0,16,True


## 2. Evolución de la aptitud y evaluaciones que traban la red

Secciones "Evolución de la aptitud" (Resultados) y "Comportamiento de convergencia" (Análisis), Tabla 5 y
recomendación sobre la detención temprana. Se usa la bitácora de evaluaciones: $J = 1/F$, y una evaluación traba la
red cuando quedan vehículos en la red al terminar (columna `trabada`, definida en `Controller.read_results`).

La Tabla 5 también la calcula la función `trabadas_por_bloque()` de `video/datos.py`; aquí se repite junto con el
dato de la generación 0.

In [6]:
ev = pd.read_csv(f"{RUN}.evaluations.csv")
ev["J"] = 1 / ev.fitness
ev["evaluacion"] = np.arange(1, len(ev) + 1)
print("evaluaciones:", len(ev), "| generaciones:", ev.generation.min(), "a", ev.generation.max())
display(ev.head())

mejor_hasta = ev.groupby("generation").J.min().cummin()
display(mejor_hasta.loc[[0, 50, 100, 126, 150]].round(2).to_frame("J del mejor individuo hasta esa generacion"))

evaluaciones: 4380 | generaciones: 0 a 150


,sid,generation,fitness,duration_mean,waiting_mean,time_loss,speed,sin_llegar,trabada,pasos,vehicles,J,evaluacion
0,0,0,0.001139,495.370,347.942,424.79,5.27,0,False,NaN,2002.0,878.106411,1
1,1,0,0.000530,998.326,808.300,927.78,3.57,0,False,NaN,2002.0,1887.455716,2
2,2,0,0.000089,5361.588,5299.662,4835.49,2.95,1047,True,NaN,1937.0,11191.216837,3
3,3,0,0.000216,2279.537,2136.939,2090.74,3.71,333,True,NaN,2000.0,4630.170516,4
4,4,0,0.000203,2412.112,2274.986,2110.33,3.85,395,True,NaN,1974.0,4914.596558,5


,J del mejor individuo hasta esa generacion
generation,
0,878.11
50,473.83
100,430.80
126,428.80
150,428.80


In [7]:
S = "2. Evolucion"
J0, J50, J100, Jfin = mejor_hasta[0], mejor_hasta[50], mejor_hasta[100], mejor_hasta[150]
verificar(S, "J generacion 0 (s)", 878.1, J0, 1)
verificar(S, "J generacion 50 (s)", 473.8, J50, 1)
verificar(S, "J generacion 100 (s)", 430.8, J100, 1)
verificar(S, "J generacion 150 (s)", 428.8, Jfin, 1)
verificar(S, "generacion del mejor individuo", 126, ev.loc[ev.J.idxmin(), "generation"], 0)
verificar(S, "reduccion de J en las primeras 50 generaciones (s)", 404.3, J0 - J50, 1)
verificar(S, "reduccion total de J (s)", 449.3, J0 - Jfin, 1)
verificar(S, "parte de la reduccion obtenida en 50 generaciones (%)", 90, 100 * (J0 - J50) / (J0 - Jfin), 0)
verificar(S, "mejora en las ultimas 50 generaciones (s)", 2.0, J100 - Jfin, 1)
verificar(S, "J de la generacion 50 sobre la solucion final (%)", 10.5, 100 * (J50 / Jfin - 1), 1)
tabla(S)

,cifra,tesis,calculado,redondeado,coincide
0,J generacion 0 (s),878.1,878.106411,878.1,True
1,J generacion 50 (s),473.8,473.833574,473.8,True
2,J generacion 100 (s),430.8,430.796442,430.8,True
3,J generacion 150 (s),428.8,428.803740,428.8,True
4,generacion del mejor individuo,126.0,126.000000,126.0,True
5,reduccion de J en las primeras 50 generaciones (s),404.3,404.272837,404.3,True
6,reduccion total de J (s),449.3,449.302671,449.3,True
7,parte de la reduccion obtenida en 50 generaciones (%),90.0,89.977840,90.0,True
8,mejora en las ultimas 50 generaciones (s),2.0,1.992703,2.0,True
9,J de la generacion 50 sobre la solucion final (%),10.5,10.501269,10.5,True


In [8]:
tramo = pd.cut(ev.generation, bins=[-1, 0, 50, 100, 150], labels=["0", "1 a 50", "51 a 100", "101 a 150"])
t5 = ev.groupby(tramo, observed=True).agg(evaluaciones=("sid", "size"), traban=("trabada", "sum"))
t5.loc["Total"] = t5.sum()
t5["proporcion_pct"] = (100 * t5.traban / t5.evaluaciones).round(1)
display(t5)

gen0 = ev[ev.generation == 0]
print("Generacion 0: la evaluacion 1 (sid 0) es la configuracion de netconvert; traba la red:", bool(gen0.iloc[0].trabada))
print("Individuos al azar de la generacion 0 que traban la red:", int(gen0.iloc[1:].trabada.sum()), "de", len(gen0) - 1)

S = "2. Evolucion"
for nombre, tesis_n, tesis_p in [("0", 18, 60.0), ("1 a 50", 102, 7.0), ("51 a 100", 44, 3.0),
                                 ("101 a 150", 87, 6.0), ("Total", 251, 5.7)]:
    verificar(S, f"Tabla 5, generaciones {nombre}: traban la red", tesis_n, t5.loc[nombre, "traban"], 0)
    verificar(S, f"Tabla 5, generaciones {nombre}: proporcion (%)", tesis_p,
              100 * t5.loc[nombre, "traban"] / t5.loc[nombre, "evaluaciones"], 1)
verificar(S, "individuos al azar de la generacion 0 que traban la red", 18, gen0.iloc[1:].trabada.sum(), 0)
verificar(S, "la configuracion de netconvert traba la red (0 = no)", 0, int(gen0.iloc[0].trabada), 0)
tabla(S).tail(12)

,evaluaciones,traban,proporcion_pct
generation,,,
0,30,18,60.0
1 a 50,1450,102,7.0
51 a 100,1450,44,3.0
101 a 150,1450,87,6.0
Total,4380,251,5.7


Generacion 0: la evaluacion 1 (sid 0) es la configuracion de netconvert; traba la red: False
Individuos al azar de la generacion 0 que traban la red: 18 de 29


,cifra,tesis,calculado,redondeado,coincide
10,"Tabla 5, generaciones 0: traban la red",18.0,18.000000,18.0,True
11,"Tabla 5, generaciones 0: proporcion (%)",60.0,60.000000,60.0,True
12,"Tabla 5, generaciones 1 a 50: traban la red",102.0,102.000000,102.0,True
13,"Tabla 5, generaciones 1 a 50: proporcion (%)",7.0,7.034483,7.0,True
14,"Tabla 5, generaciones 51 a 100: traban la red",44.0,44.000000,44.0,True
15,"Tabla 5, generaciones 51 a 100: proporcion (%)",3.0,3.034483,3.0,True
16,"Tabla 5, generaciones 101 a 150: traban la red",87.0,87.000000,87.0,True
17,"Tabla 5, generaciones 101 a 150: proporcion (%)",6.0,6.000000,6.0,True
18,"Tabla 5, generaciones Total: traban la red",251.0,251.000000,251.0,True
19,"Tabla 5, generaciones Total: proporcion (%)",5.7,5.730594,5.7,True


### Mesetas de la curva de convergencia

Recomendaciones, detención temprana. Una meseta es el tramo en que el mejor valor de $J$ no cambia: su longitud son las
generaciones transcurridas entre una mejora y la siguiente. Por ejemplo, el mejor valor obtenido en la generación 35 se
mantiene hasta la 54 y mejora en la 55, una meseta de 20 generaciones.

In [9]:
mejoras = [g for g in mejor_hasta.index[1:] if mejor_hasta[g] < mejor_hasta[g - 1]]
mesetas = pd.DataFrame({"desde": [0] + mejoras[:-1], "nueva_mejora_en": mejoras})
mesetas["generaciones"] = mesetas.nueva_mejora_en - mesetas.desde
print("Generaciones con una nueva mejora:", mejoras)
display(mesetas[mesetas.generaciones >= 10])

sin_mejora, parada = 0, None
for g in mejor_hasta.index[1:]:
    sin_mejora = 0 if mejor_hasta[g] < mejor_hasta[g - 1] else sin_mejora + 1
    if sin_mejora == 10:
        parada = g
        break
print("Un criterio de diez generaciones sin mejora detendria la ejecucion en la generacion", parada,
      "| antes de la generacion 126:", parada < 126)

S = "2. Evolucion"
verificar(S, "mesetas de diez o mas generaciones que terminan en una mejora", 4, (mesetas.generaciones >= 10).sum(), 0)
verificar(S, "meseta mas larga que termina en una mejora (generaciones)", 20, mesetas.generaciones.max(), 0)
tabla(S).tail(2)

Generaciones con una nueva mejora: [2, 4, 6, 7, 10, 18, 20, 22, 33, 35, 55, 57, 60, 71, 76, 80, 86, 93, 98, 100, 108, 126]


,desde,nueva_mejora_en,generaciones
8,22,33,11
10,35,55,20
13,60,71,11
21,108,126,18


Un criterio de diez generaciones sin mejora detendria la ejecucion en la generacion 32 | antes de la generacion 126: True


,cifra,tesis,calculado,redondeado,coincide
22,mesetas de diez o mas generaciones que terminan en una mejora,4.0,4.0,4.0,True
23,meseta mas larga que termina en una mejora (generaciones),20.0,20.0,20.0,True


## 3. Comparación con netconvert, por vehículo

Secciones "Comparación con la configuración de netconvert", "Descomposición del tiempo de viaje" y "Distribución de
las mejoras", con sus interpretaciones en Análisis. Los dos `tripinfo` están escritos en orden de llegada, así que
cada vehículo se empareja consigo mismo por su `id`. Los percentiles se calculan con el método lineal de pandas, el
mismo que usa `np.percentile` en `analisis_corrida.py`.

In [10]:
def tripinfo(ruta):
    filas = [{"id": t.get("id"), "viaje": float(t.get("duration")), "espera": float(t.get("waitingTime")),
              "llegada": float(t.get("arrival"))} for t in ET.parse(ruta).getroot().iter("tripinfo")]
    return pd.DataFrame(filas).set_index("id")


net_v = tripinfo(f"detail/{RUN}.baseline.tripinfo.xml")
alg_v = tripinfo(f"detail/{RUN}.best.tripinfo.xml")
print("vehiculos:", len(net_v), len(alg_v), "| mismos ids:", set(net_v.index) == set(alg_v.index))
print("vehiculos sin llegar:", int((net_v.llegada < 0).sum()), int((alg_v.llegada < 0).sum()))

v = net_v.join(alg_v, lsuffix="_net", rsuffix="_alg")
v["cambio_viaje"] = v.viaje_alg - v.viaje_net
display(v.head())

vehiculos: 2002 2002 | mismos ids: True
vehiculos sin llegar: 0 0


,viaje_net,espera_net,llegada_net,viaje_alg,espera_alg,llegada_alg,cambio_viaje
id,,,,,,,
64,17.0,0.0,42.0,35.0,7.0,60.0,18.0
79,21.0,0.0,52.0,69.0,38.0,117.0,48.0
0,61.0,22.0,61.0,49.0,12.0,49.0,-12.0
37,58.0,0.0,73.0,81.0,16.0,96.0,23.0
145,38.0,0.0,94.0,41.0,1.0,97.0,3.0


In [11]:
S = "3. Por vehiculo"
J_net = v.viaje_net.mean() + 1.1 * v.espera_net.mean()
J_alg = v.viaje_alg.mean() + 1.1 * v.espera_alg.mean()
mov_net = (v.viaje_net - v.espera_net).mean()
mov_alg = (v.viaje_alg - v.espera_alg).mean()
verificar(S, "vehiculos", 2002, len(v), 0)
verificar(S, "J netconvert (s)", 878.11, J_net, 2)
verificar(S, "J algoritmo (s)", 428.80, J_alg, 2)
verificar(S, "Tabla 7: variacion de J (%)", -51.17, 100 * (J_alg - J_net) / J_net, 2)
verificar(S, "Resumen: reduccion de J (%)", 51.2, 100 * (J_net - J_alg) / J_net, 1)
verificar(S, "Tabla 8: variacion del tiempo en movimiento (s)", 6.21, mov_alg - mov_net, 2)
verificar(S, "Tabla 8: variacion del tiempo de espera (s)", -216.91, v.espera_alg.mean() - v.espera_net.mean(), 2)
verificar(S, "Tabla 8: variacion del viaje total (s)", -210.70, v.viaje_alg.mean() - v.viaje_net.mean(), 2)
verificar(S, "ahorro de viaje (horas-vehiculo)", 117.2, (v.viaje_net.sum() - v.viaje_alg.sum()) / 3600, 1)
verificar(S, "ahorro de espera (horas-vehiculo)", 120.6, (v.espera_net.sum() - v.espera_alg.sum()) / 3600, 1)
tabla(S)

,cifra,tesis,calculado,redondeado,coincide
0,vehiculos,2002.00,2002.000000,2002.00,True
1,J netconvert (s),878.11,878.106394,878.11,True
2,J algoritmo (s),428.80,428.803746,428.80,True
3,Tabla 7: variacion de J (%),-51.17,-51.167222,-51.17,True
4,Resumen: reduccion de J (%),51.20,51.167222,51.20,True
5,Tabla 8: variacion del tiempo en movimiento (s),6.21,6.213786,6.21,True
6,Tabla 8: variacion del tiempo de espera (s),-216.91,-216.912587,-216.91,True
7,Tabla 8: variacion del viaje total (s),-210.70,-210.698801,-210.70,True
8,ahorro de viaje (horas-vehiculo),117.20,117.171944,117.20,True
9,ahorro de espera (horas-vehiculo),120.60,120.627500,120.60,True


In [12]:
q = pd.DataFrame({
    "viaje_net": v.viaje_net.quantile([0.01, 0.05, 0.5, 0.9]),
    "viaje_alg": v.viaje_alg.quantile([0.01, 0.05, 0.5, 0.9]),
    "espera_net": v.espera_net.quantile([0.01, 0.05, 0.5, 0.9]),
    "espera_alg": v.espera_alg.quantile([0.01, 0.05, 0.5, 0.9]),
})
q.loc["max"] = [v.viaje_net.max(), v.viaje_alg.max(), v.espera_net.max(), v.espera_alg.max()]
display(q)


def var(antes, despues):
    return 100 * (despues - antes) / antes


S = "3. Por vehiculo"
verificar(S, "percentil 90 del viaje: netconvert (s)", 1613.6, q.loc[0.9, "viaje_net"], 1)
verificar(S, "percentil 90 del viaje: algoritmo (s)", 534.9, q.loc[0.9, "viaje_alg"], 1)
verificar(S, "percentil 90 del viaje: variacion (%)", -66.9, var(q.loc[0.9, "viaje_net"], q.loc[0.9, "viaje_alg"]), 1)
verificar(S, "percentil 90 de la espera: netconvert (s)", 1398.5, q.loc[0.9, "espera_net"], 1)
verificar(S, "percentil 90 de la espera: algoritmo (s)", 280.5, q.loc[0.9, "espera_alg"], 1)
verificar(S, "percentil 90 de la espera: variacion (%)", -79.9, var(q.loc[0.9, "espera_net"], q.loc[0.9, "espera_alg"]), 1)
verificar(S, "mediana del viaje: variacion (%)", -8.9, var(q.loc[0.5, "viaje_net"], q.loc[0.5, "viaje_alg"]), 1)
verificar(S, "mediana de la espera: variacion (%)", -22.4, var(q.loc[0.5, "espera_net"], q.loc[0.5, "espera_alg"]), 1)
verificar(S, "viaje mas largo: netconvert (s)", 2389, q.loc["max", "viaje_net"], 0)
verificar(S, "viaje mas largo: algoritmo (s)", 1131, q.loc["max", "viaje_alg"], 0)
verificar(S, "viaje mas largo: variacion (%)", -52.7, var(q.loc["max", "viaje_net"], q.loc["max", "viaje_alg"]), 1)
verificar(S, "percentil 1 del viaje: netconvert (s)", 27, q.loc[0.01, "viaje_net"], 0)
verificar(S, "percentil 1 del viaje: algoritmo (s)", 34, q.loc[0.01, "viaje_alg"], 0)
verificar(S, "percentil 5 del viaje: netconvert (s)", 70.1, q.loc[0.05, "viaje_net"], 1)
verificar(S, "percentil 5 del viaje: algoritmo (s)", 73.0, q.loc[0.05, "viaje_alg"], 1)
tabla(S).tail(15)

,viaje_net,viaje_alg,espera_net,espera_alg
0.01,27.01,34.01,0.0,0.00
0.05,70.05,73.00,4.0,6.05
0.5,265.50,242.00,122.5,95.00
0.9,1613.60,534.90,1398.5,280.50
max,2389.00,1131.00,2206.0,863.00


,cifra,tesis,calculado,redondeado,coincide
10,percentil 90 del viaje: netconvert (s),1613.6,1613.600000,1613.6,True
11,percentil 90 del viaje: algoritmo (s),534.9,534.900000,534.9,True
12,percentil 90 del viaje: variacion (%),-66.9,-66.850521,-66.9,True
13,percentil 90 de la espera: netconvert (s),1398.5,1398.500000,1398.5,True
14,percentil 90 de la espera: algoritmo (s),280.5,280.500000,280.5,True
15,percentil 90 de la espera: variacion (%),-79.9,-79.942796,-79.9,True
16,mediana del viaje: variacion (%),-8.9,-8.851224,-8.9,True
17,mediana de la espera: variacion (%),-22.4,-22.448980,-22.4,True
18,viaje mas largo: netconvert (s),2389.0,2389.000000,2389.0,True
19,viaje mas largo: algoritmo (s),1131.0,1131.000000,1131.0,True


In [13]:
acortan = v[v.cambio_viaje < 0]
alargan = v[v.cambio_viaje > 0]
igual = v[v.cambio_viaje == 0]
display(pd.DataFrame({
    "vehiculos": [len(acortan), len(alargan), len(igual)],
    "viaje_medio_con_netconvert_s": [acortan.viaje_net.mean(), alargan.viaje_net.mean(), igual.viaje_net.mean()],
    "cambio_medio_s": [acortan.cambio_viaje.mean(), alargan.cambio_viaje.mean(), 0.0],
    "cambio_total_s": [acortan.cambio_viaje.sum(), alargan.cambio_viaje.sum(), 0.0],
}, index=["acortan su viaje", "alargan su viaje", "sin cambio"]))
display(alargan.cambio_viaje.describe().to_frame("aumento del viaje de los que alargan (s)"))

S = "3. Por vehiculo"
verificar(S, "vehiculos que acortan su viaje", 1311, len(acortan), 0)
verificar(S, "vehiculos que alargan su viaje", 670, len(alargan), 0)
verificar(S, "vehiculos sin cambio", 21, len(igual), 0)
verificar(S, "parte que alarga su viaje (%)", 33.5, 100 * len(alargan) / len(v), 1)
verificar(S, "Abstract: parte que alarga su viaje (%)", 33, 100 * len(alargan) / len(v), 0)
verificar(S, "aumento medio de los que alargan (s)", 54.3, alargan.cambio_viaje.mean(), 1)
verificar(S, "aumento mediano de los que alargan (s)", 42, alargan.cambio_viaje.median(), 0)
verificar(S, "aumento maximo de los que alargan (s)", 280, alargan.cambio_viaje.max(), 0)
verificar(S, "viaje medio con netconvert de los que alargan (s)", 197.8, alargan.viaje_net.mean(), 1)
verificar(S, "viaje medio con netconvert de los que acortan (s)", 653.6, acortan.viaje_net.mean(), 1)
verificar(S, "segundos adicionales de los que alargan", 36357, alargan.cambio_viaje.sum(), 0)
verificar(S, "segundos ahorrados por los que acortan", 458176, -acortan.cambio_viaje.sum(), 0)
verificar(S, "adicionales sobre ahorrados (%)", 7.9, 100 * alargan.cambio_viaje.sum() / -acortan.cambio_viaje.sum(), 1)
tabla(S).tail(13)

,vehiculos,viaje_medio_con_netconvert_s,cambio_medio_s,cambio_total_s
acortan su viaje,1311,653.559115,-349.485889,-458176.0
alargan su viaje,670,197.788060,54.264179,36357.0
sin cambio,21,114.142857,0.000000,0.0


,aumento del viaje de los que alargan (s)
count,670.000000
mean,54.264179
std,47.389589
min,1.000000
25%,18.000000
50%,42.000000
75%,79.000000
max,280.000000


,cifra,tesis,calculado,redondeado,coincide
25,vehiculos que acortan su viaje,1311.0,1311.000000,1311.0,True
26,vehiculos que alargan su viaje,670.0,670.000000,670.0,True
27,vehiculos sin cambio,21.0,21.000000,21.0,True
28,parte que alarga su viaje (%),33.5,33.466533,33.5,True
29,Abstract: parte que alarga su viaje (%),33.0,33.466533,33.0,True
30,aumento medio de los que alargan (s),54.3,54.264179,54.3,True
31,aumento mediano de los que alargan (s),42.0,42.000000,42.0,True
32,aumento maximo de los que alargan (s),280.0,280.000000,280.0,True
33,viaje medio con netconvert de los que alargan (s),197.8,197.788060,197.8,True
34,viaje medio con netconvert de los que acortan (s),653.6,653.559115,653.6,True


## 4. Distribución espacial de la mejora

Sección "Distribución espacial de la mejora" (Resultados y Análisis). El tiempo perdido de cada arista se suma sobre
todos los intervalos de la salida por arista, como en `analisis_corrida.aristas`, y las aristas se agrupan en vías con
el identificador de OpenStreetMap que netconvert conserva en su nombre (`[-]<id de la via>#<tramo>`), como en
`analisis_corrida.via`. La mitad oeste de la red son las aristas cuya coordenada x media (la de la forma de su primer
carril) es menor que el centro del recuadro de la red (`convBoundary` de `test.net.xml`).

In [14]:
def tiempo_perdido_por_arista(ruta):
    filas = [{"arista": e.get("id"), "tiempo_perdido": float(e.get("timeLoss"))}
             for e in ET.parse(ruta).getroot().iter("edge") if e.get("timeLoss") is not None]
    return pd.DataFrame(filas).groupby("arista").tiempo_perdido.sum()


ar = pd.concat({"net": tiempo_perdido_por_arista(f"detail/{RUN}.baseline.edgedata.xml"),
                "alg": tiempo_perdido_por_arista(f"detail/{RUN}.best.edgedata.xml")}, axis=1).fillna(0.0)
ar["ahorro"] = ar.net - ar.alg
ar["via"] = ar.index.str.split("#").str[0].str.lstrip("-")

x_media = {}
for e in net.findall("edge"):
    if e.get("function") != "internal":
        puntos = [tuple(map(float, p.split(","))) for p in e.find("lane").get("shape").split()]
        x_media[e.get("id")] = np.mean([p[0] for p in puntos])
ar["x"] = ar.index.map(x_media)
x0, _, x1, _ = map(float, net.find("location").get("convBoundary").split(","))
centro_x = (x0 + x1) / 2
ar["mitad"] = np.where(ar.x < centro_x, "oeste", "este")
print("aristas:", len(ar), "| sin coordenada:", int(ar.x.isna().sum()), "| centro x del recuadro:", centro_x)
display(ar.sort_values("ahorro", ascending=False).head())

aristas: 390 | sin coordenada: 0 | centro x del recuadro: 1079.825


,net,alg,ahorro,via,x,mitad
arista,,,,,,
734011349#2,49137.30,8090.57,41046.73,734011349,516.26,oeste
406701908#19,44385.37,9255.32,35130.05,406701908,533.52,oeste
384737905#9,30862.10,3429.62,27432.48,384737905,595.11,oeste
30263010#1,26482.09,4935.68,21546.41,30263010,471.78,oeste
406701908#18,29471.29,8603.67,20867.62,406701908,667.35,oeste


In [15]:
vias = ar.groupby("via")[["net", "alg", "ahorro"]].sum().sort_values("ahorro", ascending=False)
top5 = vias.head(5).copy()
top5["reduccion_pct"] = 100 * top5.ahorro / top5.net
display(top5)

en_top5 = ar[ar.via.isin(top5.index)]
por_mitad = en_top5.groupby("mitad").ahorro.sum()
display(por_mitad.to_frame("ahorro neto de las cinco vias (s)"))

positivos = ar.ahorro[ar.ahorro > 0].sort_values(ascending=False)
n_mitad = int((positivos.cumsum() < positivos.sum() / 2).sum()) + 1

S = "4. Espacial"
verificar(S, "aristas", 390, len(ar), 0)
verificar(S, "tiempo perdido total con netconvert (s)", 833295, ar.net.sum(), 0)
verificar(S, "tiempo perdido total con el algoritmo (s)", 417818, ar.alg.sum(), 0)
verificar(S, "aristas que reunen la mitad del ahorro bruto", 11, n_mitad, 0)
verificar(S, "esas aristas sobre el total (%)", 2.8, 100 * n_mitad / len(ar), 1)
verificar(S, "mayor reduccion en una arista (s)", 41047, ar.ahorro.max(), 0)
verificar(S, "mayor aumento en una arista (s)", 4646, -ar.ahorro.min(), 0)
print("mayor reduccion / mayor aumento:", round(ar.ahorro.max() / -ar.ahorro.min(), 2), "(la tesis dice 'casi nueve veces')")
print("reduccion de la via mas favorecida (%):", round(top5.reduccion_pct.iloc[0], 1), "(la tesis dice 'mas de 80%')")
verificar(S, "ahorro neto de las cinco vias en la mitad oeste (%)", 94.4, 100 * por_mitad["oeste"] / por_mitad.sum(), 1)
tabla(S)

,net,alg,ahorro,reduccion_pct
via,,,,
406701908,124699.89,20799.49,103900.40,83.320362
30279769,120626.64,36966.99,83659.65,69.354207
384737905,86653.47,23720.86,62932.61,72.625609
65777805,94908.08,46062.15,48845.93,51.466566
734011349,57073.68,9810.93,47262.75,82.810062


,ahorro neto de las cinco vias (s)
mitad,
este,19420.95
oeste,327180.39


mayor reduccion / mayor aumento: 8.84 (la tesis dice 'casi nueve veces')
reduccion de la via mas favorecida (%): 83.3 (la tesis dice 'mas de 80%')


,cifra,tesis,calculado,redondeado,coincide
0,aristas,390.0,390.000000,390.0,True
1,tiempo perdido total con netconvert (s),833295.0,833294.780000,833295.0,True
2,tiempo perdido total con el algoritmo (s),417818.0,417817.520000,417818.0,True
3,aristas que reunen la mitad del ahorro bruto,11.0,11.000000,11.0,True
4,esas aristas sobre el total (%),2.8,2.820513,2.8,True
5,mayor reduccion en una arista (s),41047.0,41046.730000,41047.0,True
6,mayor aumento en una arista (s),4646.0,4645.880000,4646.0,True
7,ahorro neto de las cinco vias en la mitad oeste (%),94.4,94.396747,94.4,True


## 5. Referencia de ciclo uniforme

Secciones "Referencia de ciclo uniforme" (Resultados) y "Comparación con una referencia de ciclo uniforme" (Análisis).
`analisis_corrida.py` guarda en `analisis.json` solo el valor de $J$ de cada ciclo. Aquí se leen las 101 simulaciones
del barrido en `outputs/`, se recalcula $J$ con la regla del controlador (un vehículo que no llega cuenta con el
horizonte completo de 10 000 s) y se marca como trabada la que deja vehículos en la red al terminar, con el mismo
criterio que `Controller.read_results`.

In [16]:
an = json.load(open(f"{RUN}.analisis.json", encoding="utf-8"))
HORIZONTE = 10000

filas = []
for c in range(20, 121):
    viajes, esperas = [], []
    for t in ET.parse(f"outputs/an_{RUN}_u{c}.xml").getroot().iter("tripinfo"):
        llego = float(t.get("arrival", 0)) >= 0
        viajes.append(float(t.get("duration")) if llego else HORIZONTE)
        esperas.append(float(t.get("waitingTime")) if llego else HORIZONTE)
    vehiculos = ET.parse(f"outputs/an_{RUN}_u{c}.stats.xml").getroot().find("vehicles")
    sin_llegar = int(vehiculos.get("running", 0)) + int(vehiculos.get("waiting", 0))
    filas.append({"ciclo_s": c, "J": round(np.mean(viajes) + 1.1 * np.mean(esperas), 2),
                  "J_en_analisis_json": an["referencia_uniforme"]["barrido"][str(c)],
                  "vehiculos_sin_llegar": sin_llegar, "traba": sin_llegar > 0})
barrido = pd.DataFrame(filas).set_index("ciclo_s")
print("J coincide con analisis.json en los 101 ciclos:", bool((barrido.J - barrido.J_en_analisis_json).abs().max() < 0.01))
print("ciclos que traban la red:", list(barrido.index[barrido.traba]))
display(barrido.loc[[50, 51, 52, 53, 54, 55, 59, 90]])

J coincide con analisis.json en los 101 ciclos: True
ciclos que traban la red: [20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 34, 40, 50, 54, 55, 60, 70, 97, 103, 105, 112, 120]


,J,J_en_analisis_json,vehiculos_sin_llegar,traba
ciclo_s,,,,
50,3931.90,3931.90,352,True
51,478.14,478.14,0,False
52,735.41,735.41,0,False
53,590.72,590.72,0,False
54,3851.41,3851.41,343,True
55,3561.31,3561.31,315,True
59,521.86,521.86,0,False
90,878.11,878.11,0,False


In [17]:
mejor_ciclo = barrido.J.idxmin()
segundo = barrido.drop(mejor_ciclo).J.idxmin()
J_uniforme = barrido.J.min()
primera = ev[ev.J < J_uniforme].iloc[0]
display(primera[["evaluacion", "sid", "generation", "J"]].to_frame("primera evaluacion con J menor que el mejor uniforme"))

S = "5. Ciclo uniforme"
verificar(S, "ciclos evaluados", 101, len(barrido), 0)
verificar(S, "ciclos que traban la red", 23, barrido.traba.sum(), 0)
verificar(S, "ciclos entre 20 y 28 s que traban la red (de 9)", 9, barrido.loc[20:28].traba.sum(), 0)
verificar(S, "mejor ciclo uniforme (s)", 51, mejor_ciclo, 0)
verificar(S, "J del mejor ciclo uniforme (s)", 478.1, J_uniforme, 1)
verificar(S, "J con ciclo de 52 s", 735.4, barrido.loc[52, "J"], 1)
verificar(S, "J con ciclo de 53 s", 590.7, barrido.loc[53, "J"], 1)
verificar(S, "segundo mejor ciclo (s)", 59, segundo, 0)
verificar(S, "J del segundo mejor ciclo (s)", 521.9, barrido.loc[segundo, "J"], 1)
verificar(S, "J con ciclo de 90 s, igual a netconvert (s)", 878.1, barrido.loc[90, "J"], 1)
verificar(S, "reduccion del mejor uniforme frente a netconvert (%)", 45.5,
          100 * (barrido.loc[90, "J"] - J_uniforme) / barrido.loc[90, "J"], 1)
verificar(S, "J del algoritmo frente al mejor uniforme (%)", -10.3, 100 * (ev.J.min() - J_uniforme) / J_uniforme, 1)
verificar(S, "evaluacion en que el algoritmo supera al mejor uniforme", 1024, primera.evaluacion, 0)
verificar(S, "generacion de esa evaluacion", 35, primera.generation, 0)
tabla(S)

,primera evaluacion con J menor que el mejor uniforme
evaluacion,1024
sid,1023
generation,35
J,473.833574


,cifra,tesis,calculado,redondeado,coincide
0,ciclos evaluados,101.0,101.000000,101.0,True
1,ciclos que traban la red,23.0,23.000000,23.0,True
2,ciclos entre 20 y 28 s que traban la red (de 9),9.0,9.000000,9.0,True
3,mejor ciclo uniforme (s),51.0,51.000000,51.0,True
4,J del mejor ciclo uniforme (s),478.1,478.140000,478.1,True
5,J con ciclo de 52 s,735.4,735.410000,735.4,True
6,J con ciclo de 53 s,590.7,590.720000,590.7,True
7,segundo mejor ciclo (s),59.0,59.000000,59.0,True
8,J del segundo mejor ciclo (s),521.9,521.860000,521.9,True
9,"J con ciclo de 90 s, igual a netconvert (s)",878.1,878.110000,878.1,True


## 6. Verificación de los pesos (Tabla 9)

Secciones "Verificación de los pesos" (Resultados) y "Dependencia de los pesos" (Análisis). Para cada criterio se
ordenan las 4380 evaluaciones y se busca la mejor y el lugar que ocupa la solución reportada (sid 3666, generación 126).

Con $w_1 = 1$, una solución $k$ supera a la reportada $r$ cuando $T_{1,k} + w_2 T_{2,k} < T_{1,r} + w_2 T_{2,r}$. La
reportada tiene la menor espera de todas, así que solo la puede superar una solución con menor tiempo de viaje, y solo
cuando $w_2 < (T_{1,r} - T_{1,k}) / (T_{2,k} - T_{2,r})$. El umbral es el mayor de esos cocientes.

In [18]:
p = ev[["evaluacion", "sid", "generation", "duration_mean", "waiting_mean", "time_loss"]].copy()
r = p[p.sid == 3666].iloc[0]
print("menor espera de todas: sid", int(p.loc[p.waiting_mean.idxmin(), "sid"]))
menor_viaje = p[p.duration_mean < r.duration_mean].copy()
menor_viaje["w2_limite"] = (r.duration_mean - menor_viaje.duration_mean) / (menor_viaje.waiting_mean - r.waiting_mean)
display(menor_viaje)
umbral = menor_viaje.w2_limite.max()
alt = menor_viaje.loc[menor_viaje.w2_limite.idxmax()]


def mejor_y_lugar(puntaje, criterio):
    orden = puntaje.rank(method="min")
    k = puntaje.idxmin()
    return {"criterio": criterio, "mejor_sid": int(p.loc[k, "sid"]), "mejor_generacion": int(p.loc[k, "generation"]),
            "lugar_de_la_reportada": int(orden[p.sid == 3666].iloc[0])}


criterios = [mejor_y_lugar(p.duration_mean + w2 * p.waiting_mean, f"w1 = 1, w2 = {w2}")
             for w2 in [0, 0.03, 0.07, 0.0754, 0.0756, 0.1, 0.5, 1.1, 2, 10]]
criterios.append(mejor_y_lugar(p.waiting_mean, "solo espera (w1 = 0)"))
criterios.append(mejor_y_lugar(p.time_loss, "tiempo perdido, sin pesos"))
t9 = pd.DataFrame(criterios)
display(t9)

menor espera de todas: sid 3666


,evaluacion,sid,generation,duration_mean,waiting_mean,time_loss,w2_limite
2902,2903,2902,100,284.524,132.975,213.98,0.07554


,criterio,mejor_sid,mejor_generacion,lugar_de_la_reportada
0,"w1 = 1, w2 = 0",2902,100,2
1,"w1 = 1, w2 = 0.03",2902,100,2
2,"w1 = 1, w2 = 0.07",2902,100,2
3,"w1 = 1, w2 = 0.0754",2902,100,2
4,"w1 = 1, w2 = 0.0756",3666,126,1
5,"w1 = 1, w2 = 0.1",3666,126,1
6,"w1 = 1, w2 = 0.5",3666,126,1
7,"w1 = 1, w2 = 1.1",3666,126,1
8,"w1 = 1, w2 = 2",3666,126,1
9,"w1 = 1, w2 = 10",3666,126,1


In [19]:
def lugar(criterio):
    return t9.set_index("criterio").loc[criterio, "lugar_de_la_reportada"]


S = "6. Pesos"
verificar(S, "umbral de w2", 0.0755, umbral, 4)
verificar(S, "umbral como porcentaje del peso del viaje (%)", 7.6, 100 * umbral, 1)
verificar(S, "lugar de la reportada con w2 = 1.1", 1, lugar("w1 = 1, w2 = 1.1"), 0)
verificar(S, "lugar de la reportada con w2 = 0.0756", 1, lugar("w1 = 1, w2 = 0.0756"), 0)
verificar(S, "lugar de la reportada con solo espera", 1, lugar("solo espera (w1 = 0)"), 0)
verificar(S, "lugar de la reportada con w2 = 0", 2, lugar("w1 = 1, w2 = 0"), 0)
verificar(S, "lugar de la reportada con w2 = 0.0754", 2, lugar("w1 = 1, w2 = 0.0754"), 0)
verificar(S, "lugar de la reportada con tiempo perdido", 2, lugar("tiempo perdido, sin pesos"), 0)
verificar(S, "generacion de la alternativa", 100, alt.generation, 0)
verificar(S, "alternativa: viaje medio menor en (s)", 0.15, r.duration_mean - alt.duration_mean, 2)
verificar(S, "alternativa: viaje medio menor en (%)", 0.05, 100 * (r.duration_mean - alt.duration_mean) / r.duration_mean, 2)
verificar(S, "alternativa: tiempo perdido menor en (s)", 0.14, r.time_loss - alt.time_loss, 2)
verificar(S, "alternativa: tiempo perdido menor en (%)", 0.07, 100 * (r.time_loss - alt.time_loss) / r.time_loss, 2)
verificar(S, "alternativa: espera mayor en (s)", 1.95, alt.waiting_mean - r.waiting_mean, 2)
verificar(S, "alternativa: espera mayor en (%)", 1.49, 100 * (alt.waiting_mean - r.waiting_mean) / r.waiting_mean, 2)
tabla(S)

,cifra,tesis,calculado,redondeado,coincide
0,umbral de w2,0.0755,0.075540,0.0755,True
1,umbral como porcentaje del peso del viaje (%),7.6000,7.553957,7.6000,True
2,lugar de la reportada con w2 = 1.1,1.0000,1.000000,1.0000,True
3,lugar de la reportada con w2 = 0.0756,1.0000,1.000000,1.0000,True
4,lugar de la reportada con solo espera,1.0000,1.000000,1.0000,True
5,lugar de la reportada con w2 = 0,2.0000,2.000000,2.0000,True
6,lugar de la reportada con w2 = 0.0754,2.0000,2.000000,2.0000,True
7,lugar de la reportada con tiempo perdido,2.0000,2.000000,2.0000,True
8,generacion de la alternativa,100.0000,100.000000,100.0000,True
9,alternativa: viaje medio menor en (s),0.1500,0.147000,0.1500,True


## 7. Robustez (Tabla 10)

Secciones "Robustez frente a la realización del escenario" (Resultados y Análisis). Se leen las 105 simulaciones de
`robustez/g150p30m12touuniP2c2600s42.json`: 3 planes por 5 semillas de demanda por 7 semillas de SUMO. El JSON
guarda la media, la desviación, el mínimo y el máximo de cada plan; aquí se calculan los cuartiles de la Tabla 10 y
las reducciones por combinación.

In [20]:
rob = pd.DataFrame(json.load(open(f"robustez/{RUN}.json", encoding="utf-8"))["filas"])
rob["traba"] = rob.sin_terminar > 0
print("simulaciones:", len(rob))
t10 = rob.groupby("plan").agg(mediana=("J", "median"), p25=("J", lambda s: s.quantile(0.25)),
                              p75=("J", lambda s: s.quantile(0.75)), minimo=("J", "min"), maximo=("J", "max"),
                              trabadas=("traba", "sum")).loc[["netconvert", "algoritmo", "uniforme"]]
display(t10.round(1))
display(rob.groupby("semilla_demanda").vehiculos.first().to_frame("vehiculos con ruta valida"))

simulaciones: 105


,mediana,p25,p75,minimo,maximo,trabadas
plan,,,,,,
netconvert,911.4,859.5,2632.8,613.8,6751.2,9
algoritmo,555.5,527.1,631.3,428.8,981.0,0
uniforme,715.0,628.2,1111.9,478.1,5735.9,7


,vehiculos con ruta valida
semilla_demanda,
1,2020
2,2020
3,2047
4,2042
42,2002


In [21]:
S = "7. Robustez"
tesis_t10 = {"netconvert": [911.4, 859.5, 2632.8, 613.8, 6751.2, 9],
             "algoritmo": [555.5, 527.1, 631.3, 428.8, 981.0, 0],
             "uniforme": [715.0, 628.2, 1111.9, 478.1, 5735.9, 7]}
for plan, valores in tesis_t10.items():
    for col, tesis_val in zip(t10.columns, valores):
        verificar(S, f"Tabla 10, {plan}: {col}", tesis_val, t10.loc[plan, col], 0 if col == "trabadas" else 1)
verificar(S, "vehiculos con ruta valida: minimo", 2002, rob.vehiculos.min(), 0)
verificar(S, "vehiculos con ruta valida: maximo", 2047, rob.vehiculos.max(), 0)
tabla(S)

,cifra,tesis,calculado,redondeado,coincide
0,"Tabla 10, netconvert: mediana",911.4,911.360,911.4,True
1,"Tabla 10, netconvert: p25",859.5,859.545,859.5,True
2,"Tabla 10, netconvert: p75",2632.8,2632.845,2632.8,True
3,"Tabla 10, netconvert: minimo",613.8,613.830,613.8,True
4,"Tabla 10, netconvert: maximo",6751.2,6751.210,6751.2,True
5,"Tabla 10, netconvert: trabadas",9.0,9.000,9.0,True
6,"Tabla 10, algoritmo: mediana",555.5,555.470,555.5,True
7,"Tabla 10, algoritmo: p25",527.1,527.075,527.1,True
8,"Tabla 10, algoritmo: p75",631.3,631.350,631.3,True
9,"Tabla 10, algoritmo: minimo",428.8,428.800,428.8,True


In [22]:
P = rob.pivot_table(index=["semilla_demanda", "semilla_sumo"], columns="plan", values="J")
red_net = 100 * (P.netconvert - P.algoritmo) / P.netconvert
red_uni = 100 * (P.uniforme - P.algoritmo) / P.uniforme
display(pd.DataFrame({"reduccion_vs_netconvert_pct": red_net.describe(),
                      "reduccion_vs_uniforme_pct": red_uni.describe()}).round(2))
alg_42 = P.loc[42, "algoritmo"].sort_values()
display(alg_42.to_frame("J del algoritmo con la demanda 42, por semilla de SUMO"))
trabadas_net = rob[(rob.plan == "netconvert") & rob.traba].semilla_demanda.value_counts()
display(trabadas_net.to_frame("combinaciones en que netconvert traba la red, por semilla de demanda"))
med = t10.mediana

S = "7. Robustez"
verificar(S, "combinaciones en que el algoritmo supera a netconvert", 35, (red_net > 0).sum(), 0)
verificar(S, "mediana de las reducciones frente a netconvert (%)", 40.3, red_net.median(), 1)
verificar(S, "menor reduccion frente a netconvert (%)", 13.7, red_net.min(), 1)
verificar(S, "mayor reduccion frente a netconvert (%)", 89.7, red_net.max(), 1)
verificar(S, "combinaciones en que el algoritmo supera al uniforme", 31, (red_uni > 0).sum(), 0)
verificar(S, "mediana de las reducciones frente al uniforme (%)", 19.8, red_uni.median(), 1)
verificar(S, "el uniforme es mejor hasta en (%)", 11.0, -red_uni.min(), 1)
verificar(S, "J del algoritmo con demanda 42 y SUMO 23 (s)", 428.8, P.loc[(42, 23), "algoritmo"], 1)
verificar(S, "es el menor de las 35 (1 = si)", 1, int(P.algoritmo.idxmin() == (42, 23)), 0)
verificar(S, "demanda 42, otras semillas de SUMO: J minimo (s)", 453.8, alg_42.drop(23).min(), 1)
verificar(S, "demanda 42, otras semillas de SUMO: J maximo (s)", 594.5, alg_42.drop(23).max(), 1)
verificar(S, "trabadas de netconvert con la semilla de demanda 4", 7, trabadas_net.get(4, 0), 0)
verificar(S, "trabadas de netconvert con la demanda 42", 1, trabadas_net.get(42, 0), 0)
verificar(S, "reduccion entre medianas frente a netconvert (%)", 39.1, 100 * (med.netconvert - med.algoritmo) / med.netconvert, 1)
verificar(S, "reduccion entre medianas frente al uniforme (%)", 22.3, 100 * (med.uniforme - med.algoritmo) / med.uniforme, 1)
tabla(S).tail(15)

,reduccion_vs_netconvert_pct,reduccion_vs_uniforme_pct
count,35.00,35.00
mean,48.59,28.84
std,25.40,30.73
min,13.66,-11.02
25%,33.27,8.25
50%,40.30,19.76
75%,69.08,34.59
max,89.66,88.55


,"J del algoritmo con la demanda 42, por semilla de SUMO"
semilla_sumo,
23,428.80
1,453.78
6,480.25
5,484.74
4,583.90
2,593.91
3,594.49


,"combinaciones en que netconvert traba la red, por semilla de demanda"
semilla_demanda,
4,7
42,1
2,1


,cifra,tesis,calculado,redondeado,coincide
20,combinaciones en que el algoritmo supera a netconvert,35.0,35.000000,35.0,True
21,mediana de las reducciones frente a netconvert (%),40.3,40.295108,40.3,True
22,menor reduccion frente a netconvert (%),13.7,13.660134,13.7,True
23,mayor reduccion frente a netconvert (%),89.7,89.658220,89.7,True
24,combinaciones en que el algoritmo supera al uniforme,31.0,31.000000,31.0,True
25,mediana de las reducciones frente al uniforme (%),19.8,19.763393,19.8,True
26,el uniforme es mejor hasta en (%),11.0,11.023524,11.0,True
27,J del algoritmo con demanda 42 y SUMO 23 (s),428.8,428.800000,428.8,True
28,es el menor de las 35 (1 = si),1.0,1.000000,1.0,True
29,"demanda 42, otras semillas de SUMO: J minimo (s)",453.8,453.780000,453.8,True


## Resumen

Todas las comparaciones del notebook. `coincide` indica si el valor calculado, redondeado a los decimales con que
aparece en la tesis, es igual a la cifra de la tesis.

In [23]:
resumen = pd.DataFrame(REGISTRO)
print("cifras verificadas:", len(resumen), "| coinciden:", int(resumen.coincide.sum()),
      "| no coinciden:", int((~resumen.coincide).sum()))
display(resumen.groupby("seccion").coincide.agg(["size", "sum"]).rename(columns={"size": "cifras", "sum": "coinciden"}))
display(resumen[~resumen.coincide])

cifras verificadas: 149 | coinciden: 149 | no coinciden: 0


,cifras,coinciden
seccion,,
1. Red,15,15
2. Evolucion,24,24
3. Por vehiculo,38,38
4. Espacial,8,8
5. Ciclo uniforme,14,14
6. Pesos,15,15
7. Robustez,35,35


,seccion,cifra,tesis,calculado,redondeado,coincide
